# Audio Source Separation Optimizer 


In [6]:
# !pip install librosa
# !pip install stempeg
# Put this in terminal if stempeg fails to pip install: $ conda install -c conda-forge ffmpeg
# !pip install musdb

from tqdm import tqdm
import stempeg
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd
from pathlib import Path

### Import MUSDB18 Dataset



In [11]:
mus18_path_tr = Path("MUSDB18/train")
mus18_path_te = Path("MUSDB18/test")
mus18_tr = list(mus18_path_tr.glob("*.mp4"))
mus18_te = list(mus18_path_te.glob("*.mp4"))
print(f"Training tracks: {len(mus18_tr)}")
print(f"Test tracks: {len(mus18_te)}")


Training tracks: 100
Test tracks: 50


#### stem_id mapping & extraction: 
 sr = 44100 \
 0 = mixture (full mix) \
 1 = drums \
 2 = bass \
 3 = other (everything else - guitars, piano, synths, etc.) \
 4 = vocals 

    access: data_stereo_tr [ i ][ j ][ k ]: 

        i = [0,1,2,3,4] stem_id mapping 
        j = [0,1] left channel, right channel
        k = song_id
    
### Audio Mono 

#### Quick Access STFT function

In [15]:
def get_stft_data(song, n_fft=2048, hop_length=1024, stereo=False, return_sample_rate=False ):
    audio, sr = stempeg.read_stems(str(song), stem_id=[0, 1, 2, 3, 4])
    stems_stft = []
    for stem_idx in range(5):
        if stereo:
            D = np.array([np.abs(librosa.stft(audio[stem_idx][:, ch], 
                                               n_fft=n_fft, 
                                               hop_length=hop_length)) 
                         for ch in range(2)])
        else:
            stem_mono = librosa.to_mono(audio[stem_idx].T)
            D = np.abs(librosa.stft(stem_mono, n_fft=n_fft, hop_length=hop_length))
        stems_stft.append(D)

    if return_sample_rate:
        return np.array(stems_stft), sr
    else:
        return np.array(stems_stft)
        


print("\nTesting STFT computation...")
test_data = get_stft_data(mus18_tr[0])
print(f"Shape: {test_data.shape}")
print(f"Sample rate: {sr}")
print("Success!")


Testing STFT computation...
Shape: (5, 1025, 11056)
Sample rate: 44100
Success!
